# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aliakhtar1010/search-ranking-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method Choice

My lane is a ranking/scoring problem built around a binary decline proxy. I will train two supervised classification models:

1. **Logistic Regression** as a simple and interpretable learned baseline.
2. **Random Forest** as a stronger nonlinear model that can capture interactions between webpage signals.

The models will output the probability that a webpage belongs to the declining class. I will use those probabilities as ranking scores rather than only making yes/no predictions.

The main evaluation metric remains **Precision@K**, because the practical goal is to place useful refresh-review candidates near the top of a limited human-review queue.

Both learned models will be compared against my frozen Week-4 `stale_but_visible` rule on the same held-out webpages and with the same Precision@K calculations. Model complexity will only be considered useful if it produces a meaningful improvement over the simpler baseline.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import sklearn

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "aliakhtar1010/search-ranking-ml/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

# Evaluation label only.
# trend_direction and trend_pct will NOT be model features.
df["declining_label"] = (
    df["trend_direction"].str.lower() == "down"
).astype(int)

print("Dataset shape:", df.shape)
print("Declining pages:", df["declining_label"].sum())
print("Base decline rate:", round(df["declining_label"].mean(), 3))
print("scikit-learn version:", sklearn.__version__)

Dataset shape: (30000, 45)
Declining pages: 16262
Base decline rate: 0.542
scikit-learn version: 1.6.1


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split Design

I use a **client-grouped holdout split** rather than randomly splitting individual webpages.

Pages belonging to the same client may share content strategy, audience, search behavior, or measurement patterns. If pages from the same client appeared in both training and testing, the evaluation could be overly optimistic because the model would be tested on groups it had partly seen before.

I therefore split using `client_id` as the group. Entire clients are assigned either to training or testing, never both.

The random seed is fixed at `42` for reproducibility.

The Week-4 baseline will also be evaluated only on this same held-out test set so that the baseline and learned models are compared fairly.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Safe numeric features available before the outcome.
# Explicitly exclude trend_direction, trend_pct, IDs,
# and the direct last30/prev30 windows used to construct trend signals.

FEATURES = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

# Make sure all requested columns exist
missing_columns = [c for c in FEATURES if c not in df.columns]
print("Missing requested feature columns:", missing_columns)

X = df[FEATURES].copy()
y = df["declining_label"].copy()
groups = df["client_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print(
    "Client overlap:",
    len(train_clients.intersection(test_clients))
)
print("Train decline rate:", round(y_train.mean(), 3))
print("Test decline rate:", round(y_test.mean(), 3))

Missing requested feature columns: []
Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0
Train decline rate: 0.55
Test decline rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

### Training and Comparison

The Week-4 rule is frozen and recreated without changing its thresholds:

- stale if `days_since_last_update >= 90`
- visible if `impressions_90d >= 731`
- qualifying pages are ranked by `impressions_90d`

I train Logistic Regression and Random Forest using the same training split. Their predicted decline probabilities become ranking scores.

All three methods are then evaluated on the exact same held-out test webpages using the same Precision@K values.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ----------------------------
# Logistic Regression
# ----------------------------

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=RANDOM_STATE
    ))
])

logistic_model.fit(X_train, y_train)

logistic_scores = logistic_model.predict_proba(X_test)[:, 1]


# ----------------------------
# Random Forest
# ----------------------------

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_scores = rf_model.predict_proba(X_test)[:, 1]


# ----------------------------
# Frozen Week-4 baseline
# ----------------------------

STALE_THRESHOLD = 90
VISIBILITY_THRESHOLD = 731

baseline_scores = np.where(
    (
        test_df["days_since_last_update"] >= STALE_THRESHOLD
    )
    &
    (
        test_df["impressions_90d"] >= VISIBILITY_THRESHOLD
    ),
    test_df["impressions_90d"],
    0
)


# ----------------------------
# Precision@K
# ----------------------------

def precision_at_k(scores, labels, k):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k_labels = labels[order[:k]]

    return top_k_labels.mean()


base_rate = y_test.mean()

results = []

methods = {
    "Week-4 baseline": baseline_scores,
    "Logistic Regression": logistic_scores,
    "Random Forest": rf_scores
}

for name, scores in methods.items():
    results.append({
        "method": name,
        "Precision@10": precision_at_k(scores, y_test, 10),
        "Precision@20": precision_at_k(scores, y_test, 20),
        "Precision@50": precision_at_k(scores, y_test, 50)
    })

results_df = pd.DataFrame(results)

results_df["Base rate"] = base_rate

results_df

,method,Precision@10,Precision@20,Precision@50,Base rate
0,Week-4 baseline,0.3,0.20,0.30,0.510952
1,Logistic Regression,0.6,0.75,0.72,0.510952
2,Random Forest,0.5,0.50,0.56,0.510952


In [4]:
import json
import os

model_metrics = {
    "random_state": RANDOM_STATE,
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "train_clients": int(len(train_clients)),
    "test_clients": int(len(test_clients)),
    "client_overlap": int(len(train_clients.intersection(test_clients))),
    "test_base_rate": float(base_rate),

    "baseline_precision_at_10": float(
        precision_at_k(baseline_scores, y_test, 10)
    ),
    "baseline_precision_at_20": float(
        precision_at_k(baseline_scores, y_test, 20)
    ),
    "baseline_precision_at_50": float(
        precision_at_k(baseline_scores, y_test, 50)
    ),

    "logistic_precision_at_10": float(
        precision_at_k(logistic_scores, y_test, 10)
    ),
    "logistic_precision_at_20": float(
        precision_at_k(logistic_scores, y_test, 20)
    ),
    "logistic_precision_at_50": float(
        precision_at_k(logistic_scores, y_test, 50)
    ),

    "random_forest_precision_at_10": float(
        precision_at_k(rf_scores, y_test, 10)
    ),
    "random_forest_precision_at_20": float(
        precision_at_k(rf_scores, y_test, 20)
    ),
    "random_forest_precision_at_50": float(
        precision_at_k(rf_scores, y_test, 50)
    )
}

os.makedirs("work/outputs", exist_ok=True)

with open("work/outputs/model_metrics.json", "w") as f:
    json.dump(model_metrics, f, indent=2)

print(model_metrics)

{'random_state': 42, 'train_rows': 23837, 'test_rows': 6163, 'train_clients': 25, 'test_clients': 7, 'client_overlap': 0, 'test_base_rate': 0.5109524582184002, 'baseline_precision_at_10': 0.3, 'baseline_precision_at_20': 0.2, 'baseline_precision_at_50': 0.3, 'logistic_precision_at_10': 0.6, 'logistic_precision_at_20': 0.75, 'logistic_precision_at_50': 0.72, 'random_forest_precision_at_10': 0.5, 'random_forest_precision_at_20': 0.5, 'random_forest_precision_at_50': 0.56}


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

### Error Analysis and Interpretation

Logistic Regression produced the strongest ranking performance on the held-out client groups, particularly at Precision@20 and Precision@50.

However, the metric alone is not enough to trust the model. I inspect which features the model relies on and examine concrete high-confidence mistakes. This helps check whether the model learned plausible relationships or is relying on a suspicious proxy for the outcome.

I also compare the model's strongest false positives and false negatives to understand where the ranking can fail. The output should therefore be treated as decision support for prioritizing human review rather than proof that a page requires a refresh.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ----------------------------
# Logistic Regression coefficients
# ----------------------------

logistic_estimator = logistic_model.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": logistic_estimator.coef_[0]
})

coef_df["absolute_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Top Logistic Regression features:")
display(coef_df.head(10))


# ----------------------------
# Build test-set prediction table
# ----------------------------

error_df = test_df[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "ctr",
        "avg_position",
        "trend_direction",
        "declining_label"
    ]
].copy()

error_df["predicted_probability"] = logistic_scores

error_df["predicted_label"] = (
    error_df["predicted_probability"] >= 0.5
).astype(int)


# ----------------------------
# False positives
# Model strongly predicts decline,
# but page is not declining
# ----------------------------

false_positives = error_df[
    (error_df["predicted_label"] == 1) &
    (error_df["declining_label"] == 0)
].sort_values(
    "predicted_probability",
    ascending=False
)

print("\nHigh-confidence false positives:")
display(false_positives.head(3))


# ----------------------------
# False negatives
# Page is declining,
# but model predicts otherwise
# ----------------------------

false_negatives = error_df[
    (error_df["predicted_label"] == 0) &
    (error_df["declining_label"] == 1)
].sort_values(
    "predicted_probability",
    ascending=True
)

print("\nHigh-confidence false negatives:")
display(false_negatives.head(3))


# ----------------------------
# Error counts
# ----------------------------

total_predictions = len(error_df)

fp_count = len(false_positives)
fn_count = len(false_negatives)

print("\nTest pages:", total_predictions)
print("False positives:", fp_count)
print("False negatives:", fn_count)

Top Logistic Regression features:


,feature,coefficient,absolute_coefficient
9,users_90d,-1.185749,1.185749
8,sessions_90d,0.965058,0.965058
13,days_with_impressions,0.661962,0.661962
14,days_with_sessions,-0.429550,0.429550
15,content_age_days,-0.407125,0.407125
3,word_count,0.321613,0.321613
12,scroll_events_90d,0.283451,0.283451
4,char_count,-0.263298,0.263298
7,pageviews_90d,0.183909,0.183909
16,days_since_last_update,0.160283,0.160283



High-confidence false positives:


,content_id,days_since_last_update,impressions_90d,ctr,avg_position,trend_direction,declining_label,predicted_probability,predicted_label
10175,content_374e795aab68,20,235,0.85,31.0,stable,0,0.866905,1
8016,content_c94a53e3bfb8,20,2164,0.23,8.1,up,0,0.865646,1
27993,content_26d48a980581,106,1266,0.00,4.6,up,0,0.864896,1



High-confidence false negatives:


,content_id,days_since_last_update,impressions_90d,ctr,avg_position,trend_direction,declining_label,predicted_probability,predicted_label
17127,content_8818fd6d967f,104,83603,1.06,3.4,down,1,0.061880,0
24849,content_2f002563e9cd,20,17,0.00,5.2,down,1,0.098092,0
26413,content_bdd7c88a58ed,14,64718,0.84,3.1,down,1,0.112965,0



Test pages: 6163
False positives: 1285
False negatives: 1309


### Interpretation

Logistic Regression produced the strongest ranking performance on the held-out clients, reaching Precision@20 of 0.75 and Precision@50 of 0.72 compared with 0.20 and 0.30 for the Week-4 rule baseline.

The strongest model coefficients were associated with `users_90d`, `sessions_90d`, and `days_with_impressions`. `days_since_last_update`, which was central to the Week-4 baseline, had a smaller coefficient than several traffic and activity signals. This suggests that the learned ranking uses multiple signals rather than relying mainly on content staleness.

The coefficient directions should be interpreted as associations within the fitted model, not causal effects. For example, a positive coefficient does not mean that increasing a feature causes content to decline.

The error review also shows that the model is not reliable for every individual page. Some pages received decline probabilities above 0.86 despite being stable or improving, while some genuinely declining pages received probabilities below 0.12. On the full held-out test set there were 1,285 false positives and 1,309 false negatives at a 0.5 classification threshold.

However, the goal of this lane is prioritization rather than perfect classification of every page. The stronger Precision@K results indicate that the model is more useful near the top of the ranked review queue than the fixed Week-4 rule. I would therefore use the output as decision support for content review rather than an automatic refresh decision.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.